# LSTM Sequence Model

This notebook implements Steps 3 to 9 with a real LSTM sequence-classification pipeline. It expects TensorFlow to be available in a Python 3.12 environment.

## Step 3: Confirm the time column and load the raw data
The model uses `timestamp` only to order events per entity.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.lstm_sequence_model import load_raw_sequence_data

df = load_raw_sequence_data()
print(df['timestamp'].dtype)
print(df[['entity_id', 'timestamp', 'label']].head())

datetime64[us]
  entity_id           timestamp   label
0     U0001 2026-01-01 08:56:33  Normal
1     U0001 2026-01-02 10:40:57  Normal
2     U0001 2026-01-03 08:10:52  Normal
3     U0001 2026-01-04 08:07:02  Normal
4     U0001 2026-01-05 09:34:42  Normal


## Step 4: Build entity-ordered sequences and labels
Each sequence is the last 5 events for an entity, padded on the left when needed. The label is the most recent event in the window.

In [2]:
from src.lstm_sequence_model import build_sequence_windows

sequences, targets, feature_columns, numeric_feature_columns, metadata = build_sequence_windows(df, window_size=5)

print('Sequences:', sequences.shape)
print('Targets:', targets.shape)
print('Feature count:', len(feature_columns))
print('Numeric feature count:', len(numeric_feature_columns))

Sequences: (45000, 5, 57)
Targets: (45000,)
Feature count: 57
Numeric feature count: 13


## Steps 5 to 7: Split, build, and train the LSTM
This uses a Masking layer, LSTM, Dropout, and a sigmoid output for binary attack prediction.

In [3]:
import importlib

import src.lstm_sequence_model as lstm_sequence_model

importlib.reload(lstm_sequence_model)

results = lstm_sequence_model.train_lstm_sequence_model(window_size=5, epochs=8, batch_size=64)

print(results['classification_report'])
print(results['confusion_matrix'])
print('ROC-AUC:', results['roc_auc'])
print('Threshold info:', results['threshold_info'])

Epoch 1/8
450/450 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.8657 - auc: 0.9094 - loss: 0.3790 - precision: 0.1050 - recall: 0.7631 - val_accuracy: 0.9007 - val_auc: 0.9823 - val_loss: 0.1729 - val_precision: 0.1688 - val_recall: 0.9932
Epoch 2/8
450/450 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9068 - auc: 0.9826 - loss: 0.1585 - precision: 0.1711 - recall: 0.9564 - val_accuracy: 0.8925 - val_auc: 0.9855 - val_loss: 0.1650 - val_precision: 0.1587 - val_recall: 1.0000
Epoch 3/8
450/450 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9011 - auc: 0.9873 - loss: 0.1299 - precision: 0.1656 - recall: 0.9808 - val_accuracy: 0.8996 - val_auc: 0.9861 - val_loss: 0.1373 - val_precision: 0.1672 - val_recall: 0.9932
Epoch 4/8
450/450 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9047 - auc: 0.9884 - loss: 0.1153 - precision: 0.1710 - recall: 0.9826 - val_accuracy: 0.8972 - val_auc: 0.9860 - val_loss: 0.1521 - val_precision: 0.1640 - val_recall: 0.9932
Epoch 5/8
450/450 ━━━━━━━━━━━━━

## Step 8: Evaluate the model
The notebook prints a classification report, confusion matrix, and ROC-AUC for the LSTM detector.

## Step 9: Saved artifacts
The trained model is saved to `trained_models/lstm_model.keras`, with feature metadata and scaling information saved alongside it.

In [4]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras

from src.baseline_profiling import create_baseline_profile_artifact
from src.lstm_sequence_model import (
    build_sequence_windows,
    load_raw_sequence_data,
)

# -----------------------------
# Load dataset and calculate attack percentage
# -----------------------------
dataset_path = ROOT / "data" / "raw" / "cybersecurity_dataset.csv"
df = pd.read_csv(dataset_path)

# Adjust this according to your labels
if df["label"].dtype == object:
    attack_percentage = (df["label"] != "Normal").mean() * 100
else:
    attack_percentage = (df["label"] != 0).mean() * 100

print(f"Attack percentage: {attack_percentage:.2f}%")

# -----------------------------
# Baseline Profiles
# -----------------------------
baseline_profiles = create_baseline_profile_artifact()
print("Baseline profiles:", len(baseline_profiles))

# -----------------------------
# Load LSTM Model
# -----------------------------
lstm_model = keras.models.load_model(
    ROOT / "trained_models" / "lstm_model.keras"
)

lstm_threshold = joblib.load(
    ROOT / "trained_models" / "lstm_threshold.pkl"
)["threshold"]

# -----------------------------
# Load sequence data
# -----------------------------
seq_df = load_raw_sequence_data()

sequences, targets, feature_columns_lstm, numeric_feature_columns, metadata = (
    build_sequence_windows(seq_df, window_size=5)
)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    sequences,
    targets,
    test_size=0.2,
    stratify=targets,
    random_state=42,
)

# -----------------------------
# LSTM Prediction
# -----------------------------
lstm_proba = lstm_model.predict(X_test_seq, verbose=0).ravel()
lstm_pred = (lstm_proba >= lstm_threshold).astype(int)

# -----------------------------
# Evaluation
# -----------------------------
print("\n========== LSTM RESULTS ==========")
print(f"Threshold : {lstm_threshold:.4f}")
print(f"Accuracy  : {accuracy_score(y_test_seq, lstm_pred):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test_seq, lstm_proba):.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test_seq, lstm_pred))

print("\nClassification Report")
print(classification_report(y_test_seq, lstm_pred, digits=4))

Attack percentage: 100.00%
Baseline profiles: 500

========== LSTM RESULTS ==========
Threshold : 0.9000
Accuracy  : 0.9807
ROC AUC   : 0.7125

Confusion Matrix
[[8820    0]
 [ 174    6]]

Classification Report
              precision    recall  f1-score   support

           0     0.9807    1.0000    0.9902      8820
           1     1.0000    0.0333    0.0645       180

    accuracy                         0.9807      9000
   macro avg     0.9903    0.5167    0.5274      9000
weighted avg     0.9810    0.9807    0.9717      9000

